In [ ]:
from google.colab import drive
from pathlib import Path

import json
import subprocess
import math
import shutil

import numpy as np
import pandas as pd
import torch

from datetime import datetime, timedelta

# 1. Mount Google Drive
drive.mount("/content/drive")

# 2. Main project directory
PROJECT = Path("/content/drive/MyDrive/CMZprojectphase2")

# 3. Input and output directories
RAW = PROJECT / "data" / "raw"
PROCESSED = PROJECT / "data" / "processed"
ANNOTATIONS = PROJECT / "data" / "annotations"

MANIFESTS = PROJECT / "manifests"
MODELS = PROJECT / "models"
OUTPUTS = PROJECT / "outputs"

# 4. Create directories if they do not exist
for folder in [
    RAW, PROCESSED, ANNOTATIONS,
    MANIFESTS, MODELS, OUTPUTS
]:
    folder.mkdir(parents=True, exist_ok=True)

# 5. Select the source video
VIDEO_PATH = RAW / "A04_20260405000000.mp4"

# 6. Check that the video exists
if not VIDEO_PATH.exists():
    raise FileNotFoundError(
        f"Video not found: {VIDEO_PATH}\n"
        "Check the filename and data/raw folder."
    )

print("Google Drive mounted.")
print("Project:", PROJECT)
print("Source video:", VIDEO_PATH)
print("Video exists:", VIDEO_PATH.exists())

Mounted at /content/drive
Google Drive mounted.
Project: /content/drive/MyDrive/CMZprojectphase2
Source video: /content/drive/MyDrive/CMZprojectphase2/data/raw/A04_20260405000000.mp4
Video exists: True


In [ ]:
%pip install -q PytorchWildlife

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from PytorchWildlife.models import detection as pw_detection

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("PyTorch-Wildlife imported successfully.")
print("Compute device:", DEVICE)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


PyTorch-Wildlife imported successfully.
Compute device: cpu


In [ ]:
from pathlib import Path
from datetime import datetime, timedelta
import subprocess
import json
import math
import shutil

import numpy as np
import pandas as pd
import torch

from PytorchWildlife.models import detection as pw_detection

# Original CCTV source
VIDEO_PATH = RAW / "A04_20260405000000.mp4"

# Recording clock confirmed by you
CCTV_RECORDING_START = datetime(2026, 4, 5, 0, 0, 0)

# Output folders on Google Drive
SCAN_DIR = OUTPUTS / "A04_full_scan_1fps"
CHUNK_RESULTS_DIR = SCAN_DIR / "completed_chunks"
CANDIDATE_CLIPS_DIR = PROCESSED / "A04_candidate_clips"

for folder in [SCAN_DIR, CHUNK_RESULTS_DIR, CANDIDATE_CLIPS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Temporary frames go on the Colab machine, not Drive.
TEMP_FRAMES_DIR = Path("/content/civet_scan_frames")

# Scan configuration
SAMPLE_INTERVAL_SECONDS = 1
CHUNK_SECONDS = 300                 # Five minutes per chunk
LOW_CONFIDENCE_THRESHOLD = 0.15     # Retain weaker detections for review
HIGH_CONFIDENCE_THRESHOLD = 0.80    # Stronger screening signal

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 8 if DEVICE == "cuda" else 4

if not VIDEO_PATH.exists():
    raise FileNotFoundError(f"Original video not found: {VIDEO_PATH}")

print("Source:", VIDEO_PATH)
print("Device:", DEVICE)
print("Batch size:", BATCH_SIZE)

# Read the source duration using ffprobe
probe = subprocess.run(
    [
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "json",
        str(VIDEO_PATH),
    ],
    capture_output=True,
    text=True,
    check=True,
)

SOURCE_DURATION = float(
    json.loads(probe.stdout)["format"]["duration"]
)

TOTAL_CHUNKS = math.ceil(SOURCE_DURATION / CHUNK_SECONDS)

print(f"Duration: {SOURCE_DURATION / 3600:.2f} hours")
print(f"Sampled frames expected: approximately {SOURCE_DURATION:.0f}")
print("Five-minute chunks:", TOTAL_CHUNKS)

Source: /content/drive/MyDrive/CMZprojectphase2/data/raw/A04_20260405000000.mp4
Device: cpu
Batch size: 4
Duration: 4.30 hours
Sampled frames expected: approximately 15471
Five-minute chunks: 52


In [ ]:
# Load the same model variant that worked in your pilot.
model = pw_detection.MegaDetectorV6Apache(
    device=DEVICE,
    pretrained=True,
    version="MDV6-apa-rtdetr-e",
)

print("MegaDetector loaded.")

frame_columns = [
    "cctv_timestamp",
    "elapsed_seconds",
    "chunk_index",
    "max_animal_confidence",
    "animal_count_ge_015",
    "animal_count_ge_080",
    "high_confidence_present",
]

detection_columns = [
    "cctv_timestamp",
    "elapsed_seconds",
    "chunk_index",
    "confidence",
    "x1", "y1", "x2", "y2",
]

for chunk_index in range(TOTAL_CHUNKS):
    chunk_start = chunk_index * CHUNK_SECONDS
    chunk_duration = min(
        CHUNK_SECONDS,
        SOURCE_DURATION - chunk_start,
    )

    frame_csv = (
        CHUNK_RESULTS_DIR /
        f"frames_chunk_{chunk_index:04d}.csv"
    )
    detection_csv = (
        CHUNK_RESULTS_DIR /
        f"detections_chunk_{chunk_index:04d}.csv"
    )

    # Skip fully completed chunks when resuming.
    if frame_csv.exists() and detection_csv.exists():
        print(f"Skipping completed chunk {chunk_index + 1}/{TOTAL_CHUNKS}")
        continue

    if TEMP_FRAMES_DIR.exists():
        shutil.rmtree(TEMP_FRAMES_DIR)
    TEMP_FRAMES_DIR.mkdir(parents=True)

    # Extract one frame per second from this source-video section.
    subprocess.run(
        [
            "ffmpeg",
            "-hide_banner", "-loglevel", "error", "-y",
            "-ss", str(chunk_start),
            "-i", str(VIDEO_PATH),
            "-t", str(chunk_duration),
            "-map", "0:v:0",
            "-vf", f"fps=1/{SAMPLE_INTERVAL_SECONDS}",
            "-q:v", "3",
            "-start_number", "0",
            str(TEMP_FRAMES_DIR / "frame_%06d.jpg"),
        ],
        check=True,
    )

    frame_paths = sorted(TEMP_FRAMES_DIR.glob("*.jpg"))

    if not frame_paths:
        raise RuntimeError(
            f"No frames extracted from chunk {chunk_index}."
        )

    # Batch detector API: results correspond to images in the
    # input folder. Zero-padded filenames preserve lexical order.
    batch_results = model.batch_image_detection(
        str(TEMP_FRAMES_DIR),
        batch_size=BATCH_SIZE,
    )

    if not isinstance(batch_results, (list, tuple)):
        raise TypeError(
            "Unexpected batch output type: "
            f"{type(batch_results)}. "
            "Inspect the output before continuing."
        )

    if len(batch_results) != len(frame_paths):
        raise RuntimeError(
            f"Chunk {chunk_index}: received {len(batch_results)} "
            f"results for {len(frame_paths)} frames. "
            "Stopping to avoid misaligned timestamps."
        )

    chunk_frame_rows = []
    chunk_detection_rows = []

    for frame_number, (frame_path, result) in enumerate(
        zip(frame_paths, batch_results)
    ):
        # Assumes the first extracted frame is at the chunk start,
        # then one frame per second.
        elapsed = (
            chunk_start +
            frame_number * SAMPLE_INTERVAL_SECONDS
        )

        if elapsed >= SOURCE_DURATION:
            continue

        cctv_time = (
            CCTV_RECORDING_START +
            timedelta(seconds=elapsed)
        ).isoformat(sep=" ")

        detections = result.get("detections")

        if detections is None:
            raise KeyError(
                "A batch result did not contain 'detections'. "
                "Stop here and inspect the installed API's output."
            )

        boxes = np.asarray(detections.xyxy)
        if boxes.size == 0:
            boxes = np.empty((0, 4))

        scores_raw = getattr(detections, "confidence", None)
        classes_raw = getattr(detections, "class_id", None)

        scores = (
            np.asarray(scores_raw).reshape(-1)
            if scores_raw is not None
            else np.zeros(len(boxes))
        )
        classes = (
            np.asarray(classes_raw).reshape(-1)
            if classes_raw is not None
            else np.full(len(boxes), -1)
        )

        animal_scores = []

        for box, score, class_id in zip(boxes, scores, classes):
            score = float(score)
            class_id = int(class_id)

            # MegaDetector category 0 = animal.
            if class_id != 0:
                continue

            animal_scores.append(score)

            if score >= LOW_CONFIDENCE_THRESHOLD:
                x1, y1, x2, y2 = map(float, box)

                chunk_detection_rows.append({
                    "cctv_timestamp": cctv_time,
                    "elapsed_seconds": elapsed,
                    "chunk_index": chunk_index,
                    "confidence": score,
                    "x1": x1,
                    "y1": y1,
                    "x2": x2,
                    "y2": y2,
                })

        max_conf = max(animal_scores, default=0.0)

        chunk_frame_rows.append({
            "cctv_timestamp": cctv_time,
            "elapsed_seconds": elapsed,
            "chunk_index": chunk_index,
            "max_animal_confidence": max_conf,
            "animal_count_ge_015": sum(
                s >= LOW_CONFIDENCE_THRESHOLD for s in animal_scores
            ),
            "animal_count_ge_080": sum(
                s >= HIGH_CONFIDENCE_THRESHOLD for s in animal_scores
            ),
            "high_confidence_present": (
                max_conf >= HIGH_CONFIDENCE_THRESHOLD
            ),
        })

    # Save each chunk to persistent storage only after inference
    # for the entire chunk completes successfully.
    pd.DataFrame(
        chunk_frame_rows, columns=frame_columns
    ).to_csv(frame_csv, index=False)

    pd.DataFrame(
        chunk_detection_rows, columns=detection_columns
    ).to_csv(detection_csv, index=False)

    print(
        f"Completed chunk {chunk_index + 1}/{TOTAL_CHUNKS}: "
        f"{len(chunk_frame_rows)} sampled frames"
    )

    # Free disk space before extracting the next chunk.
    shutil.rmtree(TEMP_FRAMES_DIR, ignore_errors=True)

print("Full-video scan finished or all chunks were already complete.")

Downloading: "https://github.com/lyuwenyu/storage/releases/download/v0.1/ResNet101_vd_ssld_pretrained_from_paddle.pth" to /root/.cache/torch/hub/checkpoints/ResNet101_vd_ssld_pretrained_from_paddle.pth


100%|██████████| 163M/163M [00:02<00:00, 77.8MB/s]


Load PResNet101 state_dict
MegaDetector loaded.
Skipping completed chunk 1/52
Skipping completed chunk 2/52
Skipping completed chunk 3/52
Completed chunk 4/52: 300 sampled frames
Completed chunk 5/52: 300 sampled frames
Completed chunk 6/52: 300 sampled frames
Completed chunk 7/52: 300 sampled frames
